# Experiment Evaluation Tables

## Goal

Find every `pt_experiment.csv` below the project's `outputs` directory and display one table per experiment. Every result row is retained, while only run identifiers, selected retrieval metrics, and their related statistical columns are shown.

## Setup

Edit `SELECTED_METRICS` to change the displayed measures. When `INCLUDE_RELATED_COLUMNS` is `True`, columns such as `ndcg_cut_10 p-value`, `ndcg_cut_10 reject`, `ndcg_cut_10 +`, and `ndcg_cut_10 -` are included automatically when present.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

# Project evaluation metrics. Add or remove names here as needed.
SELECTED_METRICS = [
    "ndcg_cut_10",
    "AP(rel=2)",
    "RR(rel=2)",
    "RR@10",
    "recall_100",
]
IDENTIFIER_COLUMNS = ["name", "run", "system", "model", "model_id", "dimension"]
INCLUDE_RELATED_COLUMNS = True
EVALUATION_FILENAME = "pt_experiment.csv"

## Discover evaluation files

The root lookup works whether the notebook is started from the repository root or from `analysis_viz`.

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the nearest parent containing the codebase directory."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "matryoshka_optimization_codebase").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from inside the Master-Thesis repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUTS_DIR = PROJECT_ROOT / "matryoshka_optimization_codebase" / "outputs"
evaluation_files = sorted(OUTPUTS_DIR.rglob(EVALUATION_FILENAME)) if OUTPUTS_DIR.exists() else []

print(f"Outputs directory: {OUTPUTS_DIR}")
print(f"Found {len(evaluation_files)} evaluation file(s).")
for evaluation_file in evaluation_files:
    print(" -", evaluation_file.relative_to(OUTPUTS_DIR))

## Results

Each CSV is loaded independently. The experiment label is its directory path relative to `outputs`, so nested experiments and repeated folder names remain distinguishable.

In [ ]:
def select_result_columns(columns: list[str]) -> list[str]:
    """Keep identifiers, selected metrics, and optional PyTerrier statistic columns."""
    identifier_columns = [column for column in IDENTIFIER_COLUMNS if column in columns]

    metric_columns = []
    for column in columns:
        for metric in SELECTED_METRICS:
            is_exact_metric = column == metric
            is_related_column = INCLUDE_RELATED_COLUMNS and column.startswith(f"{metric} ")
            if is_exact_metric or is_related_column:
                metric_columns.append(column)
                break

    # dict.fromkeys removes duplicates while preserving the CSV's useful ordering.
    return list(dict.fromkeys(identifier_columns + metric_columns))


results_by_experiment: dict[str, pd.DataFrame] = {}
discovery_rows = []

if not evaluation_files:
    display(Markdown(f"> No `{EVALUATION_FILENAME}` files were found below `{OUTPUTS_DIR}`."))
else:
    for evaluation_file in evaluation_files:
        experiment = evaluation_file.parent.relative_to(OUTPUTS_DIR).as_posix()
        try:
            full_results = pd.read_csv(evaluation_file)
            selected_columns = select_result_columns(full_results.columns.tolist())
            selected_results = full_results.loc[:, selected_columns].copy()
            results_by_experiment[experiment] = selected_results

            missing_metrics = [metric for metric in SELECTED_METRICS if metric not in full_results.columns]
            discovery_rows.append(
                {
                    "experiment": experiment,
                    "rows": len(full_results),
                    "displayed_columns": ", ".join(selected_columns) or "(none)",
                    "missing_metrics": ", ".join(missing_metrics) or "(none)",
                    "status": "ok" if selected_columns else "no selected columns found",
                }
            )

            display(Markdown(f"### `{experiment}`"))
            if selected_columns:
                display(selected_results)
            else:
                display(Markdown("> None of the configured identifier or metric columns exists in this file."))
                print("Available columns:", full_results.columns.tolist())
        except Exception as exc:
            discovery_rows.append(
                {
                    "experiment": experiment,
                    "rows": None,
                    "displayed_columns": "(none)",
                    "missing_metrics": "unknown",
                    "status": f"read error: {exc}",
                }
            )
            display(Markdown(f"### `{experiment}`\n\n> Could not read `{evaluation_file.name}`: `{exc}`"))

## Checks

This compact summary confirms the number of rows retained from each experiment and reports missing configured metrics or unreadable files.

In [ ]:
discovery_summary = pd.DataFrame(
    discovery_rows,
    columns=["experiment", "rows", "displayed_columns", "missing_metrics", "status"],
)

if discovery_summary.empty:
    print("Nothing to validate until evaluation files are available.")
else:
    display(discovery_summary)